# Monthly surface fluxes and sea ice on the shelf

We extract the monthly surface fluxes, surface properties, and sea ice fields over the Southern Ocean from the coarsened CM4X budget product (1850–2099, both experiments). These are used for the shelf-averaged time series in Figure 6 and Figures S1–S2.

In [1]:
import sys

import xarray as xr

sys.path.insert(0, "../src")
from paths import outputdir
from cm4x import cm4x_budget_files, open_cm4x_budget


surface_vars = ["fsitherm", "tos", "zos", "sos", "hfsso", "siconc", "taux", "tauy", "sithick"]
southern_rows = slice(10, 50)  # yh rows spanning ~80S-59S

In [2]:
from dask.distributed import Client
from dask_jobqueue import SLURMCluster

# 8 nodes (36 cores, 160 GB each) to read the 50 files and sum the fluxes over sigma2 in parallel
cluster = SLURMCluster(cores=36, processes=4, memory="160GB", walltime="04:00:00",
                       queue="scavenger", log_directory="logs")
cluster.scale(jobs=8)
cluster.wait_for_workers(2)
client = Client(cluster)
client

<Client: 'tcp://10.151.1.98:35241' processes=4 threads=36, memory=149.00 GiB>

In [3]:
ds = open_cm4x_budget(cm4x_budget_files(first_year=1850))[surface_vars].sel(yh=southern_rows)

# fsitherm and hfsso are split across the sigma2 classes the surface water occupies each month;
# store their monthly total, which is what the figures use (averaging per class first biases the annual mean)
ds = ds.assign({v: ds[v].sum("sigma2_l", min_count=1) for v in ["fsitherm", "hfsso"]}).drop_dims("sigma2_l")
print(f"{ds.nbytes / 1024**3:.2f} GB")

3.96 GB


In [4]:
ds.compute().to_zarr(outputdir("Southern_Ocean_50S_Monthly_Surface_Fluxes_And_Properties.zarr"), mode="w")

/user/anthony.meza/miniforge3/envs/cm4x_chapter2/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
